# External AMP Screening with AIPAMPDS

This notebook checks whether cVAE-generated peptides still look convincing under an **external** screening model rather than only under an in-repo classifier.

Why AIPAMPDS is useful here:
- it is independent from the generator training code in this repository;
- it reports `AMP`, `E. coli`, `S. aureus`, and hemolysis scores in one pass;
- it lets us inspect both **selectivity** and **safety**.

Scope and caveat:
- the platform supports only antibacterial-style readouts, so this notebook focuses on `Antibacterial`, `Gram+`, `Gram-`, and their combinations;
- these predictions are a screening signal, not experimental validation.


## Workflow

1. Load the saved cVAE checkpoint and vocabulary.
2. Generate candidates for the externally supported conditions.
3. Export FASTA and metadata for upload to AIPAMPDS.
4. Load the downloaded screening CSV and merge it back with the generated sequences.
5. Summarize condition control, rank candidates, and interpret the external results.


In [55]:
import math
import os
import pickle
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import Markdown, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid", context="talk")

cwd = Path(os.getcwd()).resolve()
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parent.parent
elif (cwd / "data").exists() and (cwd / "models").exists() and (cwd / "notebooks").exists():
    ROOT = cwd
else:
    ROOT = cwd.parent
MODELS_DIR = ROOT / "models"
DATA_DIR = ROOT / "data"
AIPAMPDS_DIR = DATA_DIR / "aipampds"
AIPAMPDS_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOKS_DIR = ROOT / "notebooks"

VOCAB_PATH = MODELS_DIR / "vocab.pkl"

# Use only the cVAE checkpoint for external screening.
CVAE_CKPT_PATH = MODELS_DIR / "best_cvae.pt"

EXPORT_FASTA_PATH = AIPAMPDS_DIR / "aipampds_sequences.fasta"
EXPORT_METADATA_PATH = AIPAMPDS_DIR / "aipampds_sequences_metadata.csv"
EXPORT_SUMMARY_PATH = AIPAMPDS_DIR / "aipampds_generation_summary.csv"
METRICS_PATH = AIPAMPDS_DIR / "aipampds_condition_metrics.csv"
RANKED_PATH = AIPAMPDS_DIR / "aipampds_ranked_candidates.csv"

AIPAMPDS_RESULTS_PATH = AIPAMPDS_DIR / "aipampds_results.csv"

AIPAMPDS_URL = "https://aipampds.pianlab.team/"

# AIPAMPDS accepts only standard amino acids within this length range.
AIPAMPDS_MIN_LEN = 5
AIPAMPDS_MAX_LEN = 50
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

GENERATOR_COND_ORDER = [
    "is_antibacterial",
    "is_anti_gram_positive",
    "is_anti_gram_negative",
    "is_antifungal",
    "is_antiviral",
    "is_antiparasitic",
    "is_anticancer",
]

# External conditions used in this notebook:
# Antibacterial -> broad antibacterial profile
# Gram+ -> shift toward Gram-positive activity
# Gram- -> shift toward Gram-negative activity
# Antibacterial + Gram+ -> broad antibacterial + Gram-positive preference
# Antibacterial + Gram- -> broad antibacterial + Gram-negative preference
# Unconditional -> no conditioning, used as a baseline
PRIMARY_EXTERNAL_CONDITIONS = {
    "Antibacterial": [1, 0, 0, 0, 0, 0, 0],
    "Gram+": [0, 1, 0, 0, 0, 0, 0],
    "Gram-": [0, 0, 1, 0, 0, 0, 0],
    "Antibacterial + Gram+": [1, 1, 0, 0, 0, 0, 0],
    "Antibacterial + Gram-": [1, 0, 1, 0, 0, 0, 0],
    "Unconditional": [0, 0, 0, 0, 0, 0, 0],
}

# These labels are excluded because AIPAMPDS does not provide matching
# external scores for them in this workflow.
EXCLUDED_CONDITIONS = [
    "Antifungal",
    "Antiviral",
    "Antiparasitic",
    "Anticancer",
]

SCREENING_THRESHOLDS = {
    "amp_score": 0.50,
    "e_coli_score": 0.50,
    "s_aureus_score": 0.50,
    "hemolytic_score": 0.50,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("root:", ROOT)
print("device:", device)
print("cvae checkpoint:", CVAE_CKPT_PATH)
print("AIPAMPDS URL:", AIPAMPDS_URL)


root: /Users/dmitrijozerov/explainable-VAE-for-AMP
device: cpu
cvae checkpoint: /Users/dmitrijozerov/explainable-VAE-for-AMP/models/best_cvae.pt
AIPAMPDS URL: https://aipampds.pianlab.team/


## 1. Load vocabulary and cVAE checkpoint

The architecture below matches the checkpoint used for generation. Nothing is trained in this notebook; the model is loaded only for inference.


In [56]:
# Fail early if the tokenizer or checkpoint is missing.
if not VOCAB_PATH.exists():
    raise FileNotFoundError(f"Missing vocabulary: {VOCAB_PATH}")
if CVAE_CKPT_PATH is None:
    raise FileNotFoundError("No cVAE checkpoint found in models/.")

# Load the saved token mapping used during training.
with open(VOCAB_PATH, "rb") as f:
    vocab_obj = pickle.load(f)

char2idx = vocab_obj["char2idx"]
idx2char = vocab_obj["idx2char"]
max_len = int(vocab_obj.get("max_len", 64))

# Special tokens needed for padding and autoregressive decoding.
PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
vocab_size = len(char2idx)

# These hyperparameters must match the trained checkpoint.
num_layers = 1
dropout = 0.2
latent_dim = 32
enc_hidden_dim = 1024
dec_hidden_dim = 512
embed_dim = 128
cond_dim = 7

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim + cond_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim + cond_dim, latent_dim)

    def forward(self, x, lengths, cond):
        # Encode the sequence, then append the condition vector before
        # predicting the latent distribution parameters.
        emb = self.drop(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        hidden = h_n[-1]
        hidden_cond = torch.cat([hidden, cond], dim=1)
        return self.fc_mu(hidden_cond), self.fc_logvar(hidden_cond)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.init_h = nn.Linear(latent_dim + cond_dim, hidden_dim)
        self.init_c = nn.Linear(latent_dim + cond_dim, hidden_dim)
        self.lstm = nn.LSTM(embed_dim + latent_dim + cond_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, z, cond):
        # Decode token by token while conditioning on both z and the label vector.
        emb = self.drop(self.embedding(x))
        z_expand = z.unsqueeze(1).expand(-1, emb.size(1), -1)
        cond_expand = cond.unsqueeze(1).expand(-1, emb.size(1), -1)
        z_cond = torch.cat([z, cond], dim=1)
        dec_inp = torch.cat([emb, z_expand, cond_expand], dim=-1)
        h0 = torch.tanh(self.init_h(z_cond)).unsqueeze(0)
        c0 = torch.tanh(self.init_c(z_cond)).unsqueeze(0)
        out, _ = self.lstm(dec_inp, (h0, c0))
        return self.fc_out(out)

class CVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hidden, dec_hidden, latent_dim, cond_dim, pad_idx):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, enc_hidden, latent_dim, cond_dim, pad_idx)
        self.decoder = Decoder(vocab_size, embed_dim, dec_hidden, latent_dim, cond_dim, pad_idx)
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim

    def reparameterize(self, mu, logvar):
        # Sample z from the learned Gaussian distribution.
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, src, lengths, cond):
        mu, logvar = self.encoder(src, lengths, cond)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(src[:, :-1], z, cond)
        target = src[:, 1:]
        return logits, target, mu, logvar

# Recreate the model architecture, then load the trained weights into it.
cvae = CVAE(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    enc_hidden=enc_hidden_dim,
    dec_hidden=dec_hidden_dim,
    latent_dim=latent_dim,
    cond_dim=cond_dim,
    pad_idx=PAD_IDX,
).to(device)

ckpt = torch.load(CVAE_CKPT_PATH, map_location=device)
state_dict = ckpt.get("model_state_dict", ckpt)
cvae.load_state_dict(state_dict)

# Switch to inference mode before generation.
cvae.eval()

print("Loaded cVAE checkpoint successfully.")
print("Vocabulary size:", vocab_size)
print("Maximum generation length:", max_len)


Loaded cVAE checkpoint successfully.
Vocabulary size: 24
Maximum generation length: 64


## 2. Generation helpers

We sample peptides only for the conditions that AIPAMPDS can score meaningfully. The helper functions below decode from latent space, filter invalid sequences, and keep a deduplicated set per condition.


In [57]:
# Build a binary condition vector in the exact order expected by the generator.
# Missing labels default to 0.
# Example:
# condition_vector(is_antibacterial=1, is_anti_gram_positive=1)
# -> [1, 1, 0, 0, 0, 0, 0]
def condition_vector(**kwargs):
    return [int(kwargs.get(k, 0)) for k in GENERATOR_COND_ORDER]

# Manual autoregressive decoding from a latent sample z and condition vector.
# The decoder starts from SOS, predicts one token at a time, and stops at EOS
# or when max_gen_len is reached.
def decode_from_z_cond(model, z, cond_t, max_gen_len=64, temperature=0.8):
    model.eval()

    # Concatenate latent and condition information once, then reuse it at
    # every decoding step so the generated sequence stays condition-aware.
    z_cond = torch.cat([z, cond_t], dim=1)
    h = torch.tanh(model.decoder.init_h(z_cond)).unsqueeze(0)
    c = torch.tanh(model.decoder.init_c(z_cond)).unsqueeze(0)

    token = torch.full((z.size(0), 1), SOS_IDX, dtype=torch.long, device=z.device)
    finished = torch.zeros(z.size(0), dtype=torch.bool, device=z.device)
    sequences = [[] for _ in range(z.size(0))]

    for _ in range(max_gen_len):
        emb = model.decoder.embedding(token)
        dec_inp = torch.cat([emb, z_cond.unsqueeze(1)], dim=-1)
        out, (h, c) = model.decoder.lstm(dec_inp, (h, c))

        # Temperature controls diversity:
        # lower -> more conservative, higher -> more random.
        logits = model.decoder.fc_out(out[:, -1, :]) / max(temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        token = next_token

        for i, idx in enumerate(next_token.squeeze(1).tolist()):
            if finished[i]:
                continue
            if idx == EOS_IDX:
                finished[i] = True
            elif idx not in (PAD_IDX, SOS_IDX):
                sequences[i].append(idx2char[idx])

        if bool(finished.all()):
            break

    return ["".join(seq) for seq in sequences]

@torch.no_grad()
def generate_cvae(model, cond, n_samples=128, temperature=0.8, alpha=1.0):
    # Repeat one condition vector across the whole batch and sample fresh z
    # values so we get many different sequences for the same target profile.
    cond_arr = np.asarray(cond, dtype=np.float32) * float(alpha)
    cond_t = torch.tensor(cond_arr, dtype=torch.float32, device=device).unsqueeze(0).repeat(n_samples, 1)
    z = torch.randn(n_samples, model.latent_dim, device=device)
    return decode_from_z_cond(model, z, cond_t, max_gen_len=max_len, temperature=temperature)

def normalize_condition_name(name):
    # Convert condition names into safe lowercase identifiers for files and ids.
    name = name.lower().replace("+", "plus")
    return re.sub(r"[^a-z0-9]+", "_", name).strip("_")

def is_aipampds_compatible(seq):
    # Keep only sequences that satisfy the external screen format constraints.
    if not seq:
        return False
    if not (AIPAMPDS_MIN_LEN <= len(seq) <= AIPAMPDS_MAX_LEN):
        return False
    return set(seq).issubset(VALID_AA)

def generate_condition_frame(model, condition_name, cond_vec, n_keep=250, batch_size=256, temperature=0.8, alpha=1.0, max_rounds=8):
    # Generate batches until we collect enough unique AIPAMPDS-compatible
    # sequences or hit the round limit.
    rows = []
    seen = set()
    raw_total = 0
    compatible_total = 0
    rounds = 0

    while len(rows) < n_keep and rounds < max_rounds:
        rounds += 1
        batch = generate_cvae(model, cond=cond_vec, n_samples=batch_size, temperature=temperature, alpha=alpha)
        raw_total += len(batch)

        for seq in batch:
            compatible = is_aipampds_compatible(seq)
            compatible_total += int(compatible)
            if not compatible:
                continue
            if seq in seen:
                continue
            seen.add(seq)
            rows.append({
                "condition": condition_name,
                "sequence": seq,
                "length": len(seq),
                "generator_condition": cond_vec,
            })
            if len(rows) >= n_keep:
                break

    frame = pd.DataFrame(rows)
    summary = {
        "condition": condition_name,
        "requested_n": int(n_keep),
        "kept_n": int(len(frame)),
        "raw_generated_n": int(raw_total),
        "compatible_raw_n": int(compatible_total),
        "compatibility_rate": float(compatible_total / raw_total) if raw_total else np.nan,
        "deduplicated_keep_rate": float(len(frame) / raw_total) if raw_total else np.nan,
        "rounds": int(rounds),
    }
    return frame, summary

demo = generate_cvae(cvae, PRIMARY_EXTERNAL_CONDITIONS["Gram+"], n_samples=5, temperature=0.8)
demo


['GLLSLLSLLGKLL',
 'FLPLIGKLLSGLS',
 'VKIIEIVVIFTGKR',
 'FLPILTNLLSGLL',
 'GFGCPNDYSCSNHCQSISGRAGYCDAATRWTRTLCTDCNGQIILTTL']

## 3. Generate and export peptides for AIPAMPDS

This cell creates:
- a deduplicated metadata CSV;
- a FASTA file ready for upload to AIPAMPDS;
- a generation summary table.

Use this cell only when you want a **new upload batch**. The downloaded AIPAMPDS CSV must match the exported FASTA and metadata files from the same run.


In [ ]:
# Generation settings for each requested external condition.
N_KEEP_PER_CONDITION = 250
GEN_BATCH_SIZE = 256
GEN_TEMPERATURE = 0.8
GEN_ALPHA = 1.0

condition_frames = []
summary_rows = []

# Generate one candidate pool per condition and keep a short summary.
for condition_name, cond_vec in PRIMARY_EXTERNAL_CONDITIONS.items():
    frame, summary = generate_condition_frame(
        cvae,
        condition_name=condition_name,
        cond_vec=cond_vec,
        n_keep=N_KEEP_PER_CONDITION,
        batch_size=GEN_BATCH_SIZE,
        temperature=GEN_TEMPERATURE,
        alpha=GEN_ALPHA,
    )
    condition_frames.append(frame)
    summary_rows.append(summary)

generated_df = pd.concat(condition_frames, ignore_index=True)

# Add stable ids so the exported FASTA can be matched back to metadata later.
generated_df["condition_slug"] = generated_df["condition"].map(normalize_condition_name)
generated_df["record_index"] = generated_df.groupby("condition").cumcount() + 1
generated_df["record_id"] = generated_df.apply(
    lambda row: f"{row.condition_slug}__{int(row.record_index):04d}", axis=1
)

summary_df = pd.DataFrame(summary_rows).sort_values("condition").reset_index(drop=True)
summary_df.to_csv(EXPORT_SUMMARY_PATH, index=False)
generated_df.to_csv(EXPORT_METADATA_PATH, index=False)

# Export the same candidates in FASTA format for AIPAMPDS upload.
fasta_lines = []
for row in generated_df.itertuples(index=False):
    fasta_lines.append(f">{row.record_id}|{row.condition}\n{row.sequence}\n")
EXPORT_FASTA_PATH.write_text("".join(fasta_lines), encoding="utf-8")

display(summary_df)
display(generated_df.head())
print("Saved FASTA:", EXPORT_FASTA_PATH)
print("Saved metadata:", EXPORT_METADATA_PATH)
print("Saved generation summary:", EXPORT_SUMMARY_PATH)
display(Markdown(
    f"**Excluded from primary external evaluation:** {', '.join(EXCLUDED_CONDITIONS)}"
))


## 4. Load the downloaded AIPAMPDS CSV

Place the screening result at:

`data/aipampds/aipampds_results.csv`

The parser is intentionally defensive about column names and merges results back by sequence plus occurrence index, which makes duplicate sequences safe to handle.


In [ ]:

# Normalize column names so small naming differences in the downloaded CSV
# do not break the parser.
def normalize_name(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")

def pick_column(df, candidates, required=True):
    norm_map = {normalize_name(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in norm_map:
            return norm_map[candidate]
    if required:
        raise KeyError(f"Could not find any of the columns: {candidates}")
    return None

def load_generated_metadata(path):
    # Reload saved generation metadata when generated_df is not in memory.
    if not path.exists():
        raise FileNotFoundError(f"Generated metadata file not found: {path}")
    df = pd.read_csv(path)
    required_cols = {"condition", "sequence"}
    missing = required_cols.difference(df.columns)
    if missing:
        raise KeyError(f"Missing columns in generated metadata: {sorted(missing)}")
    return df

def load_aipampds_results(path):
    # Parse the external screening CSV into a fixed internal schema.
    if not path.exists():
        raise FileNotFoundError(f"Results file not found: {path}")

    df = pd.read_csv(path)

    seq_col = pick_column(df, ["sequence", "peptide", "seq"])
    amp_col = pick_column(df, ["amp_score", "amp_probability", "amp"])
    ecoli_col = pick_column(df, ["e_coli_score", "e_coli", "ecoli_score", "ecoli"])
    saureus_col = pick_column(df, ["s_aureus_score", "s_aureus", "saureus_score", "saureus"])
    hemo_col = pick_column(df, ["hemolytic_score", "hemolytic", "hemolysis_score", "hemolysis"])
    overall_col = pick_column(df, ["score", "overall_score", "screening_score", "total_score"], required=False)

    out = pd.DataFrame({
        "sequence": df[seq_col].astype(str).str.strip(),
        "amp_score": pd.to_numeric(df[amp_col], errors="coerce"),
        "e_coli_score": pd.to_numeric(df[ecoli_col], errors="coerce"),
        "s_aureus_score": pd.to_numeric(df[saureus_col], errors="coerce"),
        "hemolytic_score": pd.to_numeric(df[hemo_col], errors="coerce"),
    })

    if overall_col is not None:
        out["overall_score"] = pd.to_numeric(df[overall_col], errors="coerce")

    # Keep an occurrence index so duplicate sequences can still be matched.
    out = out.dropna(subset=["sequence"]).reset_index(drop=True)
    out["sequence_occurrence"] = out.groupby("sequence").cumcount()
    return out

def ensure_results_loaded(force_reload=False, verbose=True):
    global results_df, merged_df, scored_df, generated_join_df

    # Reuse cached results unless the caller asks for a reload.
    if (
        not force_reload
        and "scored_df" in globals()
        and isinstance(scored_df, pd.DataFrame)
        and len(scored_df) > 0
    ):
        return scored_df

    if not AIPAMPDS_RESULTS_PATH.exists():
        if verbose:
            display(Markdown(
                f"**AIPAMPDS results are not loaded yet.** Put the downloaded CSV at `{AIPAMPDS_RESULTS_PATH}` and re-run this cell."
            ))
        return None

    results_df = load_aipampds_results(AIPAMPDS_RESULTS_PATH)

    if (
        "generated_df" in globals()
        and isinstance(generated_df, pd.DataFrame)
        and len(generated_df) > 0
    ):
        generated_join_df = generated_df.copy()
    elif EXPORT_METADATA_PATH.exists():
        generated_join_df = load_generated_metadata(EXPORT_METADATA_PATH)
    else:
        raise FileNotFoundError(
            "Generated metadata is missing. Run the export cell or restore "
            f"`{EXPORT_METADATA_PATH}` before loading AIPAMPDS scores."
        )

    # Merge external scores back onto the generated candidates.
    generated_join_df["sequence_occurrence"] = generated_join_df.groupby("sequence").cumcount()
    merged_df = generated_join_df.merge(results_df, on=["sequence", "sequence_occurrence"], how="left")
    scored_df = merged_df.dropna(subset=["amp_score", "e_coli_score", "s_aureus_score", "hemolytic_score"]).copy()

    if verbose:
        print("Merged rows:", len(merged_df))
        print("Rows with screening scores:", int(merged_df["amp_score"].notna().sum()))
        display(merged_df.head())

    return scored_df

scored_df = ensure_results_loaded()


## 5. Condition-level metrics for AIPAMPDS

The summary below focuses on a small set of interpretable quantities:
- `all_requested_hit_rate`: how often the requested external scores cross the threshold;
- `non_hemolytic_rate`: how often hemolysis stays below the threshold;
- `safe_hit_rate`: how often both happen at once.

For Gram-specific conditions, we also keep the non-requested species score so we can measure selectivity, not only raw activity.


In [ ]:
# Define which scores count as requested activity and which count as
# off-target activity for each condition.
CONDITION_SCORE_PLAN = {
    "Antibacterial": {
        "requested": ["amp_score", "e_coli_score", "s_aureus_score"],
        "offtarget": [],
    },
    "Gram+": {
        "requested": ["amp_score", "s_aureus_score"],
        "offtarget": ["e_coli_score"],
    },
    "Gram-": {
        "requested": ["amp_score", "e_coli_score"],
        "offtarget": ["s_aureus_score"],
    },
    "Antibacterial + Gram+": {
        "requested": ["amp_score", "s_aureus_score"],
        "offtarget": ["e_coli_score"],
    },
    "Antibacterial + Gram-": {
        "requested": ["amp_score", "e_coli_score"],
        "offtarget": ["s_aureus_score"],
    },
    "Unconditional": {
        "requested": [],
        "offtarget": [],
    },
}

def hit_mask(frame, cols):
    # Return True only when all requested scores pass the threshold.
    if not cols:
        return pd.Series([True] * len(frame), index=frame.index)
    mask = pd.Series([True] * len(frame), index=frame.index)
    for col in cols:
        mask &= frame[col] >= SCREENING_THRESHOLDS[col]
    return mask

def safe_mask(frame):
    # A sequence is considered safe here if hemolysis stays below threshold.
    return frame["hemolytic_score"] < SCREENING_THRESHOLDS["hemolytic_score"]

def summarize_condition_screening(frame, condition_name):
    # Aggregate the external screening results into a small set of condition-level metrics.
    plan = CONDITION_SCORE_PLAN[condition_name]
    requested_cols = plan["requested"]
    offtarget_cols = plan["offtarget"]

    requested_hit = hit_mask(frame, requested_cols)
    safe = safe_mask(frame)
    safe_hit = requested_hit & safe

    out = {
        "condition": condition_name,
        "n_scored": int(len(frame)),
        "all_requested_hit_rate": float(requested_hit.mean()),
        "non_hemolytic_rate": float(safe.mean()),
        "safe_hit_rate": float(safe_hit.mean()),
        "mean_amp_score": float(frame["amp_score"].mean()),
        "mean_e_coli_score": float(frame["e_coli_score"].mean()),
        "mean_s_aureus_score": float(frame["s_aureus_score"].mean()),
        "mean_hemolytic_score": float(frame["hemolytic_score"].mean()),
        "mean_requested_score": float(frame[requested_cols].mean(axis=1).mean()) if requested_cols else np.nan,
        "mean_offtarget_score": float(frame[offtarget_cols].mean(axis=1).mean()) if offtarget_cols else np.nan,
    }
    if "overall_score" in frame.columns:
        out["mean_overall_score"] = float(frame["overall_score"].mean())
    return out

def ensure_condition_metrics(force_recompute=False, verbose=True):
    global condition_metrics_df, scored_df

    scored_frame = ensure_results_loaded(verbose=verbose)
    if scored_frame is None:
        return None

    if (
        not force_recompute
        and "condition_metrics_df" in globals()
        and isinstance(condition_metrics_df, pd.DataFrame)
        and len(condition_metrics_df) > 0
    ):
        return condition_metrics_df

    metrics_rows = []
    for condition_name in PRIMARY_EXTERNAL_CONDITIONS:
        frame = scored_frame.loc[scored_frame["condition"] == condition_name].copy()
        metrics_rows.append(summarize_condition_screening(frame, condition_name))
    condition_metrics_df = pd.DataFrame(metrics_rows).sort_values("condition").reset_index(drop=True)
    condition_metrics_df.to_csv(METRICS_PATH, index=False)

    if verbose:
        display(condition_metrics_df)
        print("Saved metrics:", METRICS_PATH)

    return condition_metrics_df

condition_metrics_df = ensure_condition_metrics()


## 6. Rank the best external candidates

Ranking is heuristic, not ground truth. The score below rewards the requested activity, penalizes hemolysis, and subtracts off-target species activity for Gram-specific settings. Its purpose is shortlisting, not final selection.


In [ ]:
# Heuristic ranking score for candidate triage.
# Reward requested activity, reward low hemolysis, and penalize off-target scores.
def composite_candidate_score(frame, condition_name):
    plan = CONDITION_SCORE_PLAN[condition_name]
    requested_cols = plan["requested"]
    offtarget_cols = plan["offtarget"]

    requested_mean = frame[requested_cols].mean(axis=1) if requested_cols else pd.Series(np.zeros(len(frame)), index=frame.index)
    offtarget_mean = frame[offtarget_cols].mean(axis=1) if offtarget_cols else pd.Series(np.zeros(len(frame)), index=frame.index)
    safety_bonus = 1.0 - frame["hemolytic_score"]
    return 0.60 * requested_mean + 0.30 * safety_bonus - 0.10 * offtarget_mean

def ensure_ranked_candidates(force_recompute=False, verbose=True):
    global ranked_candidates_df

    scored_frame = ensure_results_loaded(verbose=verbose)
    if scored_frame is None:
        return None

    if (
        not force_recompute
        and "ranked_candidates_df" in globals()
        and isinstance(ranked_candidates_df, pd.DataFrame)
        and len(ranked_candidates_df) > 0
    ):
        return ranked_candidates_df

    ranked_frames = []
    for condition_name in PRIMARY_EXTERNAL_CONDITIONS:
        frame = scored_frame.loc[scored_frame["condition"] == condition_name].copy()

        # Add binary flags used for ranking and quick inspection.
        frame["requested_hit"] = hit_mask(frame, CONDITION_SCORE_PLAN[condition_name]["requested"]).astype(int)
        frame["non_hemolytic"] = safe_mask(frame).astype(int)
        frame["safe_hit"] = (frame["requested_hit"] & frame["non_hemolytic"]).astype(int)
        frame["composite_score"] = composite_candidate_score(frame, condition_name)

        # Sort best-first and keep only a short shortlist per condition.
        frame = frame.sort_values(["safe_hit", "composite_score", "amp_score"], ascending=[False, False, False])
        ranked_frames.append(frame.head(15))

    ranked_candidates_df = pd.concat(ranked_frames, ignore_index=True)
    ranked_candidates_df.to_csv(RANKED_PATH, index=False)

    if verbose:
        display(ranked_candidates_df[[
            "condition", "record_id", "sequence", "length", "safe_hit", "composite_score",
            "amp_score", "e_coli_score", "s_aureus_score", "hemolytic_score"
        ]].head(30))
        print("Saved ranked candidates:", RANKED_PATH)

    return ranked_candidates_df

ranked_candidates_df = ensure_ranked_candidates()


## 7. Visual summary

The goal is to answer three questions with as little visual clutter as possible:
1. Which condition produces the strongest externally screened hits?
2. Do Gram-specific conditions really shift selectivity in the expected direction?
3. Where does better control start to cost hemolysis?


In [ ]:

# Load the summarized metrics and create a compact high-level view.
condition_metrics_df = ensure_condition_metrics(verbose=False)
scored_df = ensure_results_loaded(verbose=False)

if condition_metrics_df is None or scored_df is None:
    display(Markdown(
        f"**Plots are waiting for AIPAMPDS results.** Put the downloaded CSV at `{AIPAMPDS_RESULTS_PATH}` and re-run this cell."
    ))
else:
    condition_order = list(PRIMARY_EXTERNAL_CONDITIONS.keys())
    palette = dict(zip(condition_order, sns.color_palette("colorblind", n_colors=len(condition_order))))
    display_label_map = {
        "Antibacterial": "Antibacterial",
        "Gram+": "Gram+",
        "Gram-": "Gram-",
        "Antibacterial + Gram+": "AB + Gram+",
        "Antibacterial + Gram-": "AB + Gram-",
        "Unconditional": "Unconditional",
    }

    plot_df = condition_metrics_df.copy()
    plot_df["condition"] = pd.Categorical(plot_df["condition"], categories=condition_order, ordered=True)
    plot_df = plot_df.sort_values("condition").reset_index(drop=True)

    # Species gap > 0 means stronger Gram+ tendency; < 0 means stronger Gram- tendency.
    plot_df["species_gap"] = plot_df["mean_s_aureus_score"] - plot_df["mean_e_coli_score"]

    labels = plot_df["condition"].astype(str).tolist()
    display_labels = [display_label_map[label] for label in labels]
    y_pos = np.arange(len(plot_df))
    colors = ["#8a8f98" if label == "Unconditional" else palette[label] for label in labels]
    color_map = dict(zip(labels, colors))

    fig = plt.figure(figsize=(16, 9.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.05])
    axes = np.array([
        fig.add_subplot(grid[0, 0]),
        fig.add_subplot(grid[0, 1]),
        fig.add_subplot(grid[1, :]),
    ], dtype=object)

    # Panel 1: how often each condition produces externally safe hits.
    axes[0].barh(y_pos, plot_df["safe_hit_rate"], color=colors, edgecolor="white", height=0.68)
    axes[0].set_yticks(y_pos, display_labels)
    axes[0].invert_yaxis()
    axes[0].set_xlim(0, 1.0)
    axes[0].set_xlabel("Safe hit rate")
    axes[0].set_title("External screening success")
    axes[0].grid(axis="x", alpha=0.20)
    for y, value in zip(y_pos, plot_df["safe_hit_rate"]):
        axes[0].text(min(value + 0.02, 0.96), y, f"{value:.2f}", va="center", fontsize=10)
    axes[0].text(
        0.02,
        1.02,
        "Gray = unconditional baseline",
        transform=axes[0].transAxes,
        fontsize=10,
        color="#666666",
    )

    # Panel 2: trade-off between safety and condition control.
    axes[1].axvspan(0, SCREENING_THRESHOLDS["hemolytic_score"], color="#2a9d8f", alpha=0.08)
    axes[1].axvline(SCREENING_THRESHOLDS["hemolytic_score"], linestyle="--", color="#c44e52", linewidth=1.4)
    x_limit = max(0.55, float(plot_df["mean_hemolytic_score"].max()) + 0.10)
    for row in plot_df.itertuples(index=False):
        label = str(row.condition)
        axes[1].scatter(
            row.mean_hemolytic_score,
            row.safe_hit_rate,
            s=170,
            color=color_map[label],
            edgecolor="white",
            linewidth=0.9,
            zorder=3,
        )
        x_offset = -8 if row.mean_hemolytic_score > x_limit - 0.12 else 8
        y_offset = -10 if row.safe_hit_rate > 0.88 else 8
        ha = "right" if x_offset < 0 else "left"
        va = "top" if y_offset < 0 else "bottom"
        axes[1].annotate(
            display_label_map[label],
            (row.mean_hemolytic_score, row.safe_hit_rate),
            xytext=(x_offset, y_offset),
            textcoords="offset points",
            fontsize=10,
            ha=ha,
            va=va,
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.82),
        )
    axes[1].set_title("Safety vs condition control")
    axes[1].set_xlabel("Mean hemolytic score\n(lower is better)")
    axes[1].set_ylabel("Safe hit rate")
    axes[1].set_xlim(0, x_limit)
    axes[1].set_ylim(0, 1.02)
    axes[1].grid(alpha=0.20)

    # Panel 3: direction and strength of Gram selectivity.
    axes[2].axvline(0.0, linestyle="--", color="0.45", linewidth=1.2)
    axes[2].barh(y_pos, plot_df["species_gap"], color=colors, edgecolor="white", height=0.68)
    axes[2].set_yticks(y_pos, display_labels)
    axes[2].invert_yaxis()
    axes[2].set_xlabel("Species gap (S. aureus - E. coli)")
    axes[2].set_title("Direction of species selectivity")
    species_gap_limit = float(np.abs(plot_df["species_gap"]).max()) + 0.08
    axes[2].set_xlim(-species_gap_limit, species_gap_limit)
    axes[2].grid(axis="x", alpha=0.20)
    for y, value in zip(y_pos, plot_df["species_gap"]):
        ha = "left" if value >= 0 else "right"
        offset = 0.02 if value >= 0 else -0.02
        axes[2].text(value + offset, y, f"{value:+.2f}", va="center", ha=ha, fontsize=10)
    axes[2].text(
        0.02,
        1.02,
        "Positive favors Gram+, negative favors Gram-",
        transform=axes[2].transAxes,
        fontsize=10,
        color="#666666",
    )

    sns.despine(fig=fig)
    plt.show()


## 8. Figure 1: concise readout

- `Unconditional` is the baseline. Its `safe_hit_rate = 0.912` is high because no species-specific target is requested, so it is a reference for general AMP-like generation, not proof of controllability.
- `Gram+` and `Antibacterial + Gram+` are the cleanest controlled settings in this screen. Their `safe_hit_rate` values are `0.372` and `0.396`, and both show a strong positive species gap (`+0.355` and `+0.362`), which is exactly the expected direction.
- `Gram-` and `Antibacterial + Gram-` also move in the correct direction (`-0.267` and `-0.255`), but they pay more in safety: the mean hemolytic score rises to `0.443` and `0.483`.
- The plain `Antibacterial` condition is the weakest targeted setting (`safe_hit_rate = 0.128`). The model keeps peptides AMP-like, but broad support across both bacteria is inconsistent.


In [ ]:

# Build a second figure with per-sample distributions instead of condition averages.
scored_df = ensure_results_loaded(verbose=False)

if scored_df is None:
    display(Markdown(
        f"**Plots are waiting for AIPAMPDS results.** Put the downloaded CSV at `{AIPAMPDS_RESULTS_PATH}` and re-run this cell."
    ))
else:
    condition_order = list(PRIMARY_EXTERNAL_CONDITIONS.keys())
    palette = dict(zip(condition_order, sns.color_palette("colorblind", n_colors=len(condition_order))))
    display_label_map = {
        "Antibacterial": "Antibacterial",
        "Gram+": "Gram+",
        "Gram-": "Gram-",
        "Antibacterial + Gram+": "AB + Gram+",
        "Antibacterial + Gram-": "AB + Gram-",
        "Unconditional": "Unconditional",
    }
    label_order = [display_label_map[name] for name in condition_order]
    label_palette = {display_label_map[name]: palette[name] for name in condition_order}

    sampled_frames = []
    for condition_name in condition_order:
        frame = scored_df.loc[scored_df["condition"] == condition_name].copy()
        if len(frame) == 0:
            continue
        # Plot only a sample of points so the scatter stays readable.
        sampled_frames.append(frame.sample(min(len(frame), 70), random_state=SEED))

    sample_df = pd.concat(sampled_frames, ignore_index=True)
    sample_df["condition"] = pd.Categorical(sample_df["condition"], categories=condition_order, ordered=True)
    sample_df = sample_df.sort_values("condition")
    sample_df["condition_label"] = sample_df["condition"].astype(str).map(display_label_map)

    scored_plot_df = scored_df.copy()
    scored_plot_df["condition"] = pd.Categorical(scored_plot_df["condition"], categories=condition_order, ordered=True)
    scored_plot_df = scored_plot_df.sort_values("condition")
    scored_plot_df["condition_label"] = scored_plot_df["condition"].astype(str).map(display_label_map)

    fig, axes = plt.subplots(2, 1, figsize=(13.5, 9.8), constrained_layout=True)

    # Left panel: raw candidates in E. coli / S. aureus score space.
    sns.scatterplot(
        data=sample_df,
        x="e_coli_score",
        y="s_aureus_score",
        hue="condition_label",
        hue_order=label_order,
        palette=label_palette,
        s=55,
        alpha=0.55,
        ax=axes[0],
    )
    axes[0].axvline(SCREENING_THRESHOLDS["e_coli_score"], linestyle="--", color="#c44e52", linewidth=1.3)
    axes[0].axhline(SCREENING_THRESHOLDS["s_aureus_score"], linestyle="--", color="#c44e52", linewidth=1.3)
    axes[0].plot([0, 1], [0, 1], linestyle=":", color="0.55", linewidth=1.0)
    axes[0].set_title("Sampled candidates in species space")
    axes[0].set_xlabel("E. coli score")
    axes[0].set_ylabel("S. aureus score")
    axes[0].set_xlim(0, 1.02)
    axes[0].set_ylim(0, 1.02)
    axes[0].grid(alpha=0.18)
    handles, legend_labels = axes[0].get_legend_handles_labels()
    legend = axes[0].get_legend()
    if legend is not None:
        legend.remove()
    axes[0].legend(
        handles,
        legend_labels,
        title="Condition",
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=3,
        frameon=False,
    )

    # Right panel: hemolysis distribution by condition.
    sns.boxplot(
        data=scored_plot_df,
        x="hemolytic_score",
        y="condition_label",
        hue="condition_label",
        order=label_order,
        hue_order=label_order,
        palette=label_palette,
        dodge=False,
        showfliers=False,
        linewidth=1.1,
        ax=axes[1],
    )
    legend = axes[1].get_legend()
    if legend is not None:
        legend.remove()
    sns.stripplot(
        data=sample_df,
        x="hemolytic_score",
        y="condition_label",
        order=label_order,
        color="black",
        alpha=0.18,
        jitter=0.18,
        size=2.3,
        ax=axes[1],
    )
    axes[1].axvline(SCREENING_THRESHOLDS["hemolytic_score"], linestyle="--", color="#c44e52", linewidth=1.3)
    axes[1].set_title("Hemolysis distribution")
    axes[1].set_xlabel("Hemolytic score")
    axes[1].set_ylabel("")
    axes[1].set_xlim(-0.02, 1.02)
    axes[1].grid(axis="x", alpha=0.18)

    sns.despine(fig=fig)
    plt.show()


## 9. Baseline interpretation

The baseline in this notebook is `Unconditional` generation passed through the same external screen. It tells us what the decoder produces without asking for a specific target profile.

A useful way to read the baseline is:
- it is the strongest setting for generic AMP-like and non-hemolytic peptides;
- it is **not** selective, because both species scores stay moderate and close to each other;
- a conditioned result is only meaningfully better than baseline if it moves the species scores in the requested direction while keeping hemolysis under control.

One more caveat matters: the global `AMP` score is almost saturated for every condition. In practice, the informative part of this external classifier is the combination of **species-specific scores plus hemolysis**, not the AMP score alone.


## 10. Pros, cons, and future work

**Pros of this external classifier**
- independent from both the cVAE and the in-repo baseline classifier;
- fast multi-objective screening for activity and safety in one pass;
- useful for reranking large candidate batches before wet-lab follow-up.

**Cons**
- narrow label coverage: it does not evaluate antifungal, antiviral, or antiparasitic targets used elsewhere in the project;
- the global `AMP` score saturates near `1.0`, so it has limited resolution among already AMP-like peptides;
- broad-spectrum conclusions are indirect because they rely on only two bacterial species;
- it is still a prediction model, not experimental validation.

**Future work**
- compare top candidates across several external predictors instead of relying on one screen;
- add novelty, diversity, and homology filters before final ranking;
- calibrate thresholds on a held-out external benchmark rather than using only a fixed `0.5` rule;
- move from post-hoc screening to multi-objective generation that explicitly penalizes hemolysis;
- validate the top-ranked candidates experimentally.
